In [0]:
"""
Taco Bueno Data Cleaning - Delta Table Version
===============================================
Creates a managed Delta table (Databricks handles storage automatically).
Run this ONCE to create the clean dataset.
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

print("=" * 80)
print("TACO BUENO - CREATE CLEAN DELTA TABLE")
print("=" * 80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# ============================================================================
# CONFIGURATION
# ============================================================================

# Input: Your raw data volume
INPUT_PATH = '/Volumes/workspace/default/taco_bueno_raw/TB_Daily_TLD_*-2026-m.csv'

# Table name (Databricks will manage storage automatically)
DELTA_TABLE_NAME = "workspace.default.taco_bueno_clean_revision"

INCOMPLETE_STORES = ['5263', '5898']

print(f"📁 Input path: {INPUT_PATH}")
print(f"📊 Table name: {DELTA_TABLE_NAME}")
print()

# ============================================================================
# DEFINE SCHEMA
# ============================================================================

schema = StructType([
    StructField("store_number", StringType(), True),
    StructField("date", StringType(), True),
    StructField("total", StringType(), True),
    StructField("check_number", StringType(), True),
    StructField("time_card", StringType(), True),
    StructField("begin_time", StringType(), True),
    StructField("end_time", StringType(), True),
    StructField("total_discount", StringType(), True),
    StructField("order_voided", StringType(), True),
    StructField("tax", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("transaction_type", StringType(), True),
    StructField("menu_item", StringType(), True),
    StructField("menu_item_ext_ref", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("pos_code", StringType(), True),
    StructField("menu_price", StringType(), True),
    StructField("item_discount", StringType(), True),
    StructField("item_voided", StringType(), True),
    StructField("report_group", StringType(), True),
    StructField("tender_type", StringType(), True)
])

# ============================================================================
# STEP 1: READ RAW DATA
# ============================================================================

print("STEP 1: Reading raw CSV files...")
print("-" * 80)

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .schema(schema) \
    .csv(INPUT_PATH)

initial_count = df_raw.count()
print(f"✓ Loaded {initial_count:,} rows")
print()

# ============================================================================
# STEP 2: REMOVE HEADER ROWS
# ============================================================================

print("STEP 2: Removing header rows...")
print("-" * 80)

df_clean = df_raw.filter(col("store_number") != "Store #")
clean_count = df_clean.count()
print(f"✓ Removed {initial_count - clean_count:,} header rows")
print(f"  Remaining: {clean_count:,} rows")
print()

# ============================================================================
# STEP 3: DATA TYPE CONVERSIONS & CLEANING
# ============================================================================

print("STEP 3: Converting data types and cleaning...")
print("-" * 80)

df_typed = df_clean \
    .withColumn("total", col("total").cast("double")) \
    .withColumn("total_discount", col("total_discount").cast("double")) \
    .withColumn("order_voided", col("order_voided").cast("double")) \
    .withColumn("tax", col("tax").cast("double")) \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("menu_price", col("menu_price").cast("double")) \
    .withColumn("item_discount", col("item_discount").cast("double")) \
    .withColumn("item_voided", col("item_voided").cast("double"))

# Parse date
df_typed = df_typed \
    .withColumn("order_date", to_date(col("date"), "M/d/yyyy")) \
    .filter(col("order_date").isNotNull())

# Extract date components
df_typed = df_typed \
    .withColumn("year", year(col("order_date"))) \
    .withColumn("month", month(col("order_date"))) \
    .withColumn("day", dayofmonth(col("order_date"))) \
    .withColumn("day_of_week", dayofweek(col("order_date")) - 1) \
    .withColumn("day_name", date_format(col("order_date"), "EEEE")) \
    .withColumn("week_of_year", weekofyear(col("order_date"))) \
    .withColumn("is_weekend", when(col("day_of_week").isin(0, 6), 1).otherwise(0))

# Parse time to hour
df_typed = df_typed \
    .withColumn("begin_hour", 
        when(col("begin_time").rlike("^[0-9]{1,2}:[0-9]{2} [AP]M$"),
            hour(to_timestamp(col("begin_time"), "h:mm a"))
        ).otherwise(None)
    ) \
    .withColumn("end_hour",
        when(col("end_time").rlike("^[0-9]{1,2}:[0-9]{2} [AP]M$"),
            hour(to_timestamp(col("end_time"), "h:mm a"))
        ).otherwise(None)
    )

# Time period
df_typed = df_typed \
    .withColumn("time_period",
        when(col("begin_hour").between(6, 10), "Breakfast")
        .when(col("begin_hour").between(11, 13), "Lunch")
        .when(col("begin_hour").between(14, 16), "Afternoon")
        .when(col("begin_hour").between(17, 20), "Dinner")
        .otherwise("Late Night")
    )

# Clean text fields
df_typed = df_typed \
    .withColumn("menu_item", 
        coalesce(regexp_replace(col("menu_item"), '["\']', ''), lit("Unknown Item"))
    ) \
    .withColumn("channel", coalesce(trim(col("channel")), lit("Unknown"))) \
    .withColumn("transaction_type", coalesce(trim(col("transaction_type")), lit("Unknown"))) \
    .withColumn("report_group", coalesce(trim(col("report_group")), lit("Unknown"))) \
    .withColumn("tender_type", coalesce(trim(col("tender_type")), lit("Unknown")))

print("✓ Data types converted and cleaned")
print()

# ============================================================================
# STEP 4: REMOVE DUPLICATES
# ============================================================================

print("STEP 4: Removing duplicates...")
print("-" * 80)

before_dedup = df_typed.count()

df_dedup = df_typed
print("Filas conservadas; revisión de duplicados pendiente.")

after_dedup = df_dedup.count()
print(f"✓ Removed {before_dedup - after_dedup:,} duplicates")
print(f"  Remaining: {after_dedup:,} rows")
print()

# ============================================================================
# STEP 5: CREATE FLAGS AND DERIVED FIELDS
# ============================================================================

print("STEP 5: Creating flags and derived fields...")
print("-" * 80)

df_final = df_dedup \
    .withColumn("is_negative_total", when(col("total") < 0, 1).otherwise(0)) \
    .withColumn("is_negative_price", when(col("menu_price") < 0, 1).otherwise(0)) \
    .withColumn("is_zero_quantity", when(col("quantity") == 0, 1).otherwise(0)) \
    .withColumn("is_zero_price", when(col("menu_price") == 0, 1).otherwise(0)) \
    .withColumn("is_incomplete_store", when(col("store_number").isin(INCOMPLETE_STORES), 1).otherwise(0)) \
    .withColumn("net_total", lit(None).cast("double")) \
    .withColumn("net_item_price", col("quantity") * col("menu_price") - col("item_discount")) \
    .withColumn("price_per_unit",
        when(
            col("quantity") > 0,
            col("net_item_price") / col("quantity")
        ).otherwise(lit(None).cast("double"))
    ) \
    .withColumn("has_discount",
        when((col("total_discount") > 0) | (col("item_discount") > 0), 1).otherwise(0)
    ) \
    .withColumn("discount_pct",
        when(
            (col("quantity") > 0) & (col("menu_price") > 0), 100 * col("item_discount") /
            (col("quantity") * col("menu_price"))
        ).otherwise(lit(None).cast("double"))
    ) \
    .withColumn("is_voided",
        when(
            (coalesce(col("order_voided"), lit(0)) != 0) |
            (coalesce(col("item_voided"), lit(0)) != 0) |
            (lower(trim(col("transaction_type"))) == "void"),
            1
        ).otherwise(0)
    ) \
    .withColumn("item_category",
        when(col("report_group").contains("Taco"), "Tacos")
        .when(col("report_group").contains("Burrito"), "Burritos")
        .when(col("report_group").contains("Beverage") | col("report_group").contains("Drink"), "Beverages")
        .when(col("report_group").contains("Side") | col("report_group").contains("Nacho") | col("report_group").contains("Chip"), "Sides")
        .when(col("report_group").contains("Quesadilla"), "Quesadillas")
        .when(col("report_group").contains("Condiment") | col("report_group").contains("Modifier"), "Modifiers")
        .when(col("report_group").contains("Paper"), "Packaging")
        .otherwise("Other")
    ) \
    .withColumn("is_online_order",
        when(lower(col("channel")).contains("online"), 1).otherwise(0)
    )

print("✓ Created all derived fields and flags")
print()

# ============================================================================
# STEP 6: SELECT FINAL COLUMNS
# ============================================================================

print("STEP 6: Selecting final columns...")
print("-" * 80)

final_columns = [
    # Identifiers
    "store_number", "check_number", "pos_code",
    # Date/Time
    "order_date", "year", "month", "day", "day_of_week", "day_name",
    "week_of_year", "is_weekend", "begin_time", "end_time", "begin_hour",
    "end_hour", "time_period",
    # Transaction
    "transaction_type", "channel", "tender_type",
    # Item
    "menu_item", "menu_item_ext_ref", "report_group", "item_category", "quantity",
    # Pricing
    "menu_price", "item_discount", "net_item_price", "price_per_unit",
    "total", "total_discount", "net_total", "tax",
    # Flags
    "has_discount", "discount_pct", "is_voided", "order_voided", "item_voided",
    "is_negative_total", "is_negative_price", "is_zero_quantity",
    "is_zero_price", "is_incomplete_store", "is_online_order",
    # Other
    "time_card"
]

df_output = df_final.select(final_columns)

print(f"✓ Selected {len(final_columns)} columns")
print()

# ============================================================================
# STEP 7: DROP EXISTING TABLE (if any)
# ============================================================================

print("STEP 7: Preparing to create table...")
print("-" * 80)

spark.sql(f"DROP TABLE IF EXISTS {DELTA_TABLE_NAME}")
print(f"✓ Dropped existing table (if any)")
print()

# ============================================================================
# STEP 8: WRITE AS MANAGED DELTA TABLE
# ============================================================================

print("STEP 8: Writing Delta table (managed by Databricks)...")
print("-" * 80)

# Write as a managed table (Databricks handles storage)
df_output.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .option("overwriteSchema", "true") \
    .saveAsTable(DELTA_TABLE_NAME)

print(f"✓ Table created: {DELTA_TABLE_NAME}")
print()

# ============================================================================
# STEP 9: ADD TABLE DESCRIPTION
# ============================================================================

print("STEP 9: Adding table description...")
print("-" * 80)

spark.sql(f"""
    COMMENT ON TABLE {DELTA_TABLE_NAME} IS 
    'Clean Taco Bueno transaction data (Jan-Sep 2026). 
    Partitioned by year/month. 
    Use is_incomplete_store=0 to exclude stores with partial data (#5263, #5898).
    Created: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
""")

print("✓ Table description added")
print()

# ============================================================================
# STEP 10: OPTIMIZE TABLE
# ============================================================================

print("STEP 10: Optimizing table...")
print("-" * 80)

spark.sql(f"OPTIMIZE {DELTA_TABLE_NAME}")
spark.sql(f"ANALYZE TABLE {DELTA_TABLE_NAME} COMPUTE STATISTICS")

print("✓ Table optimized and statistics computed")
print()

# ============================================================================
# STEP 11: SUMMARY STATISTICS
# ============================================================================

print("STEP 11: Summary statistics...")
print("-" * 80)

# Una fila por ticket para no sumar sus importes por cada producto.
claves_ticket = ["store_number", "order_date", "check_number"]

tickets_revision = (
    spark.table(DELTA_TABLE_NAME)
    .filter(col("is_incomplete_store") == 0)
    .groupBy(*claves_ticket)
    .agg(
        countDistinct("total").alias("versiones_total"),
        countDistinct("total_discount").alias("versiones_descuento"),
        countDistinct("tax").alias("versiones_impuesto"),
        max("total").alias("total_registrado"),
        max("total_discount").alias("descuento_ticket"),
        max("tax").alias("impuesto"),
        max("has_discount").alias("con_descuento")
    )
)

# Revisar tickets cuyos importes no son constantes entre sus filas.
inconsistentes = tickets_revision.filter(
    (col("versiones_total") != 1) |
    (col("versiones_descuento") != 1) |
    (col("versiones_impuesto") != 1)
)

display(inconsistentes)

# Resumen provisional: solo tickets con importes consistentes.
tickets_consistentes = tickets_revision.filter(
    (col("versiones_total") == 1) &
    (col("versiones_descuento") == 1) &
    (col("versiones_impuesto") == 1)
)

display(tickets_consistentes.agg(
    count("*").alias("tickets_consistentes"),
    round(sum("total_registrado"), 2).alias("total_registrado"),
    round(sum("descuento_ticket"), 2).alias("descuento_de_ticket"),
    sum("con_descuento").alias("tickets_con_descuento")
))

print()

# ============================================================================
# STEP 12: SHOW TABLE LOCATION
# ============================================================================

print("STEP 12: Table information...")
print("-" * 80)

table_info = spark.sql(f"DESCRIBE EXTENDED {DELTA_TABLE_NAME}")
location = table_info.filter(col("col_name") == "Location").select("data_type").first()

if location:
    print(f"📁 Table location: {location[0]}")
else:
    print("📁 Table location: Managed by Databricks")

print()

# ============================================================================
# COMPLETION
# ============================================================================

print("=" * 80)
print("✅ DELTA TABLE CREATION COMPLETE!")
print("=" * 80)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()
print(f"📊 Table name: {DELTA_TABLE_NAME}")
print()
print("🎯 Your team can now use:")
print(f"   SQL: SELECT * FROM {DELTA_TABLE_NAME} WHERE month = 1")
print(f"   Python: df = spark.table('{DELTA_TABLE_NAME}')")
print(f"   Pandas: df = spark.table('{DELTA_TABLE_NAME}').filter('month = 1').toPandas()")
print()
print("📋 Next steps:")
print("   1. ✅ Data is ready for analysis")
print("   2. Share table name with your team")
print("   3. Create analysis notebooks")
print("   4. Export to CSV if needed for R")
print("=" * 80)

TACO BUENO - CREATE CLEAN DELTA TABLE
Start time: 2026-10-07 23:59:04

📁 Input path: /Volumes/workspace/default/taco_bueno_raw/TB_Daily_TLD_*-2026-m.csv
📊 Table name: workspace.default.taco_bueno_clean_revision

STEP 1: Reading raw CSV files...
--------------------------------------------------------------------------------
✓ Loaded 42,253,092 rows

STEP 2: Removing header rows...
--------------------------------------------------------------------------------
✓ Removed 0 header rows
  Remaining: 42,253,092 rows

STEP 3: Converting data types and cleaning...
--------------------------------------------------------------------------------
✓ Data types converted and cleaned

STEP 4: Removing duplicates...
--------------------------------------------------------------------------------
Filas conservadas; revisión de duplicados pendiente.
✓ Removed 0 duplicates
  Remaining: 42,253,092 rows

STEP 5: Creating flags and derived fields...
-------------------------------------------------------

store_number,order_date,check_number,versiones_total,versiones_descuento,versiones_impuesto,total_registrado,descuento_ticket,impuesto,con_descuento


tickets_consistentes,total_registrado,descuento_de_ticket,tickets_con_descuento
6012195,8.743520905E7,1574701.87,432954



STEP 12: Table information...
--------------------------------------------------------------------------------
📁 Table location: 

✅ DELTA TABLE CREATION COMPLETE!
End time: 2026-10-08 00:01:29

📊 Table name: workspace.default.taco_bueno_clean_revision

🎯 Your team can now use:
   SQL: SELECT * FROM workspace.default.taco_bueno_clean_revision WHERE month = 1
   Python: df = spark.table('workspace.default.taco_bueno_clean_revision')
   Pandas: df = spark.table('workspace.default.taco_bueno_clean_revision').filter('month = 1').toPandas()

📋 Next steps:
   1. ✅ Data is ready for analysis
   2. Share table name with your team
   3. Create analysis notebooks
   4. Export to CSV if needed for R


In [0]:
print("Tickets con importes inconsistentes:", inconsistentes.count())
display(inconsistentes.limit(20))

Tickets con importes inconsistentes: 0


store_number,order_date,check_number,versiones_total,versiones_descuento,versiones_impuesto,total_registrado,descuento_ticket,impuesto,con_descuento


In [0]:
%sql
SELECT * FROM workspace.default.taco_bueno_clean_revision LIMIT 10;

store_number,check_number,pos_code,order_date,year,month,day,day_of_week,day_name,week_of_year,is_weekend,begin_time,end_time,begin_hour,end_hour,time_period,transaction_type,channel,tender_type,menu_item,menu_item_ext_ref,report_group,item_category,quantity,menu_price,item_discount,net_item_price,price_per_unit,total,total_discount,net_total,tax,has_discount,discount_pct,is_voided,order_voided,item_voided,is_negative_total,is_negative_price,is_zero_quantity,is_zero_price,is_incomplete_store,is_online_order,time_card
6905,97148001589249,001056,2025-12-31,2025,12,31,3,Wednesday,1,0,9:31 AM,9:33 AM,9,9,Breakfast,Regular Sale,Drive Thru,Unknown,Beef Muchaco,null,(M) Tacos,Tacos,1,4.49,0.0,4.49,4.49,6.69,0.0,null,0.51,0,0.0,0,0.0,0.0,0,0,0,0,0,0,null
6905,97148001589249,001000,2025-12-31,2025,12,31,3,Wednesday,1,0,9:31 AM,9:33 AM,9,9,Breakfast,Regular Sale,Drive Thru,Unknown,Modifier (NR),null,(M) Other,Other,1,0.0,0.0,0.0,0.0,6.69,0.0,null,0.51,0,null,0,0.0,0.0,0,0,0,1,0,0,null
6905,97148001589249,001002,2025-12-31,2025,12,31,3,Wednesday,1,0,9:31 AM,9:33 AM,9,9,Breakfast,Regular Sale,Drive Thru,Unknown,Extra Tomatoes,null,(M) Condiments,Modifiers,1,0.0,0.0,0.0,0.0,6.69,0.0,null,0.51,0,null,0,0.0,0.0,0,0,0,1,0,0,null
6905,97148001589249,001096,2025-12-31,2025,12,31,3,Wednesday,1,0,9:31 AM,9:33 AM,9,9,Breakfast,Regular Sale,Drive Thru,Unknown,Party Quesadilla,null,(M) Bueno Under a Buck,Other,1,1.69,0.0,1.69,1.69,6.69,0.0,null,0.51,0,0.0,0,0.0,0.0,0,0,0,0,0,0,null
6905,97148001589249,Free Cup of Water,2025-12-31,2025,12,31,3,Wednesday,1,0,9:31 AM,9:33 AM,9,9,Breakfast,Regular Sale,Drive Thru,Unknown,Free Water (NR),null,(M) Beverages,Beverages,1,0.0,0.0,0.0,0.0,6.69,0.0,null,0.51,0,null,0,0.0,0.0,0,0,0,1,0,0,null
6905,97148001585153,001095,2025-12-31,2025,12,31,3,Wednesday,1,0,9:49 AM,9:51 AM,9,9,Breakfast,Regular Sale,Carry Out,Unknown,Party Taco,null,(M) Bueno Under a Buck,Other,1,1.69,0.0,1.69,1.69,7.32,0.0,null,0.56,0,0.0,0,0.0,0.0,0,0,0,0,0,0,null
6905,97148001585153,001095,2025-12-31,2025,12,31,3,Wednesday,1,0,9:49 AM,9:51 AM,9,9,Breakfast,Regular Sale,Carry Out,Unknown,Party Taco,null,(M) Bueno Under a Buck,Other,1,1.69,0.0,1.69,1.69,7.32,0.0,null,0.56,0,0.0,0,0.0,0.0,0,0,0,0,0,0,null
6905,97148001585153,001095,2025-12-31,2025,12,31,3,Wednesday,1,0,9:49 AM,9:51 AM,9,9,Breakfast,Regular Sale,Carry Out,Unknown,Party Taco,null,(M) Bueno Under a Buck,Other,1,1.69,0.0,1.69,1.69,7.32,0.0,null,0.56,0,0.0,0,0.0,0.0,0,0,0,0,0,0,null
6905,97148001585153,002062,2025-12-31,2025,12,31,3,Wednesday,1,0,9:49 AM,9:51 AM,9,9,Breakfast,Regular Sale,Carry Out,Unknown,Grilled Chicken Taco Rollup,null,(M) Tacos,Tacos,1,1.69,0.0,1.69,1.69,7.32,0.0,null,0.56,0,0.0,0,0.0,0.0,0,0,0,0,0,0,null
6905,97148001585154,001077,2025-12-31,2025,12,31,3,Wednesday,1,0,9:56 AM,9:57 AM,9,9,Breakfast,Regular Sale,Dine-In,Unknown,Combo 1 (NR),null,(M) Combos,Other,1,9.49,0.0,9.49,9.49,10.27,0.0,null,0.78,0,0.0,0,0.0,0.0,0,0,0,0,0,0,null


In [0]:
from pyspark.sql import functions as F

revision = spark.table("workspace.default.taco_bueno_clean_revision")

# 1. Cobertura y datos que necesitan revisión.
display(
    revision.agg(
        F.count("*").alias("filas"),
        F.min("order_date").alias("fecha_inicial"),
        F.max("order_date").alias("fecha_final"),

        F.sum(F.when(
            F.col("store_number").isNull() |
            F.col("order_date").isNull() |
            F.col("check_number").isNull(),
            1
        ).otherwise(0)).alias("filas_sin_clave_ticket"),

        F.sum(F.when(
            F.col("total").isNull() |
            F.col("total_discount").isNull() |
            F.col("tax").isNull(),
            1
        ).otherwise(0)).alias("filas_con_importes_faltantes"),

        F.sum(F.when(
            F.col("quantity").isNull() |
            (F.col("quantity") <= 0),
            1
        ).otherwise(0)).alias("filas_cantidad_nula_cero_negativa"),

        F.sum(F.when(
            ~F.col("net_total").eqNullSafe(F.col("total")),
            1
        ).otherwise(0)).alias("filas_net_total_distinto_total")
    )
)

# 2. Valores reales de las anulaciones y tipos de transacción.
display(
    revision.groupBy(
        "transaction_type", "order_voided", "item_voided"
    )
    .count()
    .orderBy(F.desc("count"))
)

# 3. Ejemplos con varias unidades para revisar el precio por unidad.
display(
    revision.filter(
        (F.col("quantity") > 1) &
        (F.col("menu_price") > 0)
    )
    .select(
        "store_number", "order_date", "check_number",
        "menu_item", "quantity", "menu_price",
        "item_discount", "net_item_price", "price_per_unit"
    )
    .limit(20)
)

filas,fecha_inicial,fecha_final,filas_sin_clave_ticket,filas_con_importes_faltantes,filas_cantidad_nula_cero_negativa,filas_net_total_distinto_total
42253092,2025-12-31,2026-09-27,0,0,74737,42253092


transaction_type,order_voided,item_voided,count
Regular Sale,0.0,0.0,42085591
Regular Sale,0.0,1.69,26817
Regular Sale,0.0,2.69,23953
Regular Sale,0.0,1.89,12506
Regular Sale,0.0,2.89,11935
Discount,0.0,0.0,10339
Regular Sale,0.0,2.99,8870
Regular Sale,0.0,3.69,7281
Regular Sale,0.0,4.49,6548
Regular Sale,0.0,4.99,4627


store_number,order_date,check_number,menu_item,quantity,menu_price,item_discount,net_item_price,price_per_unit
7910,2026-01-08,164040,Bean Burrito,2,2.69,0.0,5.38,2.69
7910,2026-01-08,164042,Party Burrito,2,1.69,0.0,3.38,1.69
7910,2026-01-08,164043,Party Burrito,3,1.69,0.0,5.07,1.6900000000000002
7910,2026-01-08,164044,Beef Burrito,2,4.29,0.0,8.58,4.29
7910,2026-01-08,164048,Beef Burrito,3,4.29,0.0,12.870000000000001,4.29
7910,2026-01-08,164055,Party Burrito,2,1.69,0.0,3.38,1.69
7910,2026-01-08,164056,Bean Burrito,3,2.69,0.0,8.07,2.69
7910,2026-01-08,164059,Bean Burrito,3,2.69,0.0,8.07,2.69
7910,2026-01-08,164060,Party Burrito,15,1.69,0.0,25.349999999999998,1.69
7910,2026-01-08,164060,Party Quesadilla,2,1.69,0.0,3.38,1.69


In [0]:
from pyspark.sql import functions as F

revision = spark.table("workspace.default.taco_bueno_clean_revision")

display(
    revision.filter(
        (F.col("store_number") == "7910") &
        (F.col("order_date") == F.lit("2026-01-08").cast("date")) &
        (F.col("check_number") == "164060")
    ).select(
        "menu_item", "quantity", "menu_price", "item_discount",
        "total", "total_discount", "tax",
        "transaction_type", "order_voided", "item_voided"
    )
)

# Separar las cantidades problemáticas por tipo de transacción.
display(
    revision.filter(
        F.col("quantity").isNull() | (F.col("quantity") <= 0)
    )
    .groupBy("transaction_type", "quantity")
    .count()
    .orderBy(F.desc("count"))
)

menu_item,quantity,menu_price,item_discount,total,total_discount,tax,transaction_type,order_voided,item_voided
Party Burrito,15,1.69,0.0,28.73,0.0,2.37,Regular Sale,0.0,0.0
Party Quesadilla,2,1.69,0.0,28.73,0.0,2.37,Regular Sale,0.0,0.0


transaction_type,quantity,count
Regular Sale,0,74737


In [0]:
from pyspark.sql import functions as F

revision = spark.table("workspace.default.taco_bueno_clean_revision")

# Muestra de tickets con descuento de producto para aclarar su significado.
claves = ["store_number", "order_date", "check_number"]

ejemplos = (
    revision.filter(
        (F.col("item_discount") != 0) &
        (F.col("quantity") > 1)
    )
    .select(*claves)
    .distinct()
    .limit(5)
)

display(
    revision.join(ejemplos, claves, "inner")
    .select(
        *claves, "menu_item", "quantity", "menu_price",
        "item_discount", "total", "total_discount", "tax",
        "transaction_type", "order_voided", "item_voided"
    )
    .orderBy(*claves)
)

store_number,order_date,check_number,menu_item,quantity,menu_price,item_discount,total,total_discount,tax,transaction_type,order_voided,item_voided
7910,2026-01-08,164114,Beef BOB,0,4.99,0.0,40.54,8.11,2.68,Regular Sale,0.0,0.0
7910,2026-01-08,164114,Modifier (NR),0,0.0,0.0,40.54,8.11,2.68,Regular Sale,0.0,0.0
7910,2026-01-08,164114,Beef BOB,0,4.99,0.0,40.54,8.11,2.68,Regular Sale,0.0,0.0
7910,2026-01-08,164114,Cheesecake 2-Pack,1,3.69,0.73,40.54,8.11,2.68,Discount,0.0,0.0
7910,2026-01-08,164114,BF Nachos,3,8.49,5.1,40.54,8.11,2.68,Discount,0.0,0.0
7910,2026-01-08,164114,Dip Cup - Chocolate Sauce,1,0.0,0.0,40.54,8.11,2.68,Regular Sale,0.0,0.0
7910,2026-01-08,164114,Chicken BOB,2,5.69,2.28,40.54,8.11,2.68,Discount,0.0,0.0
7910,2026-01-08,164117,Beef Crispy Taco,7,2.69,1.89,21.52,2.15,1.6,Discount,0.0,0.0
7910,2026-01-08,164117,Bean Burrito,1,2.69,0.26,21.52,2.15,1.6,Discount,0.0,0.0
7910,2026-01-08,442583,Combo Burrito,1,3.99,3.99,20.62,20.62,0.0,Discount,0.0,0.0


In [0]:
from pyspark.sql import functions as F

revision = spark.table("workspace.default.taco_bueno_clean_revision")

display(
    revision.agg(
        F.count("*").alias("filas"),

        F.sum(F.when(
            F.col("discount_pct") > 100.01, 1
        ).otherwise(0)).alias("descuentos_mayores_100_pct"),

        F.sum(F.when(
            F.col("discount_pct") < 0, 1
        ).otherwise(0)).alias("descuentos_negativos"),

        F.sum(F.when(
            (F.col("quantity") > 0) &
            (F.col("menu_price") > 0) &
            (F.col("net_item_price") < -0.01),
            1
        ).otherwise(0)).alias("lineas_positivas_con_neto_negativo")
    )
)

filas,descuentos_mayores_100_pct,descuentos_negativos,lineas_positivas_con_neto_negativo
42253092,0,3,0


In [0]:
display(
    revision.filter(F.col("discount_pct") < 0)
    .select(
        "store_number", "order_date", "check_number",
        "menu_item", "transaction_type",
        "quantity", "menu_price", "item_discount",
        "net_item_price", "discount_pct",
        "total", "total_discount", "tax",
        "order_voided", "item_voided", "is_voided"
    )
)

store_number,order_date,check_number,menu_item,transaction_type,quantity,menu_price,item_discount,net_item_price,discount_pct,total,total_discount,tax,order_voided,item_voided,is_voided
5898,2026-01-04,443499,Grilled Chicken Taco Rollup,Discount,3,1.69,-0.57,5.640000000000001,-11.242603550295856,0.0,1.52,1.57,0.0,0.0,0
5898,2026-01-04,443499,Party Burrito,Discount,2,1.69,-0.38,3.76,-11.242603550295858,0.0,1.52,1.57,0.0,0.0,0
5898,2026-01-04,443499,Party Taco,Discount,3,1.69,-0.57,5.640000000000001,-11.242603550295856,0.0,1.52,1.57,0.0,0.0,0


In [0]:
from pyspark.sql import functions as F

# Leer únicamente los identificadores y el archivo de origen.
origen = (
    spark.read
    .option("header", "true")
    .schema(schema)
    .csv(INPUT_PATH)
    .select(
        "store_number",
        F.to_date("date", "M/d/yyyy").alias("order_date"),
        "check_number",
        F.col("_metadata.file_path").alias("archivo_origen")
    )
    .filter(
        F.col("order_date").isNotNull() &
        F.col("store_number").isNotNull() &
        F.col("check_number").isNotNull()
    )
)

tickets_varios_archivos = (
    origen
    .groupBy("store_number", "order_date", "check_number")
    .agg(
        F.countDistinct("archivo_origen").alias("numero_archivos"),
        F.collect_set("archivo_origen").alias("archivos")
    )
    .filter(F.col("numero_archivos") > 1)
)

print(
    "Tickets presentes en varios archivos:",
    tickets_varios_archivos.count()
)
display(tickets_varios_archivos.limit(20))

Tickets presentes en varios archivos: 0


store_number,order_date,check_number,numero_archivos,archivos


In [0]:
"""
Taco Bueno Data Cleaning - Delta Table Version
===============================================
Creates a managed Delta table (Databricks handles storage automatically).
Run this ONCE to create the clean dataset.
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

print("=" * 80)
print("TACO BUENO - CREATE CLEAN DELTA TABLE")
print("=" * 80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# ============================================================================
# CONFIGURATION
# ============================================================================

# Input: Your raw data volume
INPUT_PATH = '/Volumes/workspace/default/taco_bueno_raw/TB_Daily_TLD_*-2026-m.csv'

# Table name (Databricks will manage storage automatically)
DELTA_TABLE_NAME = "workspace.default.taco_bueno_clean"

INCOMPLETE_STORES = ['5263', '5898']

print(f"📁 Input path: {INPUT_PATH}")
print(f"📊 Table name: {DELTA_TABLE_NAME}")
print()

# ============================================================================
# DEFINE SCHEMA
# ============================================================================

schema = StructType([
    StructField("store_number", StringType(), True),
    StructField("date", StringType(), True),
    StructField("total", StringType(), True),
    StructField("check_number", StringType(), True),
    StructField("time_card", StringType(), True),
    StructField("begin_time", StringType(), True),
    StructField("end_time", StringType(), True),
    StructField("total_discount", StringType(), True),
    StructField("order_voided", StringType(), True),
    StructField("tax", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("transaction_type", StringType(), True),
    StructField("menu_item", StringType(), True),
    StructField("menu_item_ext_ref", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("pos_code", StringType(), True),
    StructField("menu_price", StringType(), True),
    StructField("item_discount", StringType(), True),
    StructField("item_voided", StringType(), True),
    StructField("report_group", StringType(), True),
    StructField("tender_type", StringType(), True)
])

# ============================================================================
# STEP 1: READ RAW DATA
# ============================================================================

print("STEP 1: Reading raw CSV files...")
print("-" * 80)

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .schema(schema) \
    .csv(INPUT_PATH)

df_raw = df_raw.select("*", col("_metadata.file_path").alias("source_file"))

initial_count = df_raw.count()
print(f"✓ Loaded {initial_count:,} rows")
print()

# ============================================================================
# STEP 2: REMOVE HEADER ROWS
# ============================================================================

print("STEP 2: Removing header rows...")
print("-" * 80)

df_clean = df_raw.filter(col("store_number") != "Store #")
clean_count = df_clean.count()
print(f"✓ Removed {initial_count - clean_count:,} header rows")
print(f"  Remaining: {clean_count:,} rows")
print()

# ============================================================================
# STEP 3: DATA TYPE CONVERSIONS & CLEANING
# ============================================================================

print("STEP 3: Converting data types and cleaning...")
print("-" * 80)

df_typed = df_clean \
    .withColumn("total", col("total").cast("double")) \
    .withColumn("total_discount", col("total_discount").cast("double")) \
    .withColumn("order_voided", col("order_voided").cast("double")) \
    .withColumn("tax", col("tax").cast("double")) \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("menu_price", col("menu_price").cast("double")) \
    .withColumn("item_discount", col("item_discount").cast("double")) \
    .withColumn("item_voided", col("item_voided").cast("double"))

# Parse date
df_typed = df_typed \
    .withColumn("order_date", to_date(col("date"), "M/d/yyyy")) \
    .filter(col("order_date").isNotNull())

# Extract date components
df_typed = df_typed \
    .withColumn("year", year(col("order_date"))) \
    .withColumn("month", month(col("order_date"))) \
    .withColumn("day", dayofmonth(col("order_date"))) \
    .withColumn("day_of_week", dayofweek(col("order_date")) - 1) \
    .withColumn("day_name", date_format(col("order_date"), "EEEE")) \
    .withColumn("week_of_year", weekofyear(col("order_date"))) \
    .withColumn("is_weekend", when(col("day_of_week").isin(0, 6), 1).otherwise(0))

# Parse time to hour
df_typed = df_typed \
    .withColumn("begin_hour", 
        when(col("begin_time").rlike("^[0-9]{1,2}:[0-9]{2} [AP]M$"),
            hour(to_timestamp(col("begin_time"), "h:mm a"))
        ).otherwise(None)
    ) \
    .withColumn("end_hour",
        when(col("end_time").rlike("^[0-9]{1,2}:[0-9]{2} [AP]M$"),
            hour(to_timestamp(col("end_time"), "h:mm a"))
        ).otherwise(None)
    )

# Time period
df_typed = df_typed \
    .withColumn("time_period",
        when(col("begin_hour").isNull(), "Unknown")
        .when(col("begin_hour").between(6, 10), "Breakfast")
        .when(col("begin_hour").between(11, 13), "Lunch")
        .when(col("begin_hour").between(14, 16), "Afternoon")
        .when(col("begin_hour").between(17, 20), "Dinner")
        .otherwise("Late Night")
    )

# Clean text fields
df_typed = df_typed \
    .withColumn("menu_item", 
        coalesce(regexp_replace(col("menu_item"), '["\']', ''), lit("Unknown Item"))
    ) \
    .withColumn("channel", coalesce(trim(col("channel")), lit("Unknown"))) \
    .withColumn("transaction_type", coalesce(trim(col("transaction_type")), lit("Unknown"))) \
    .withColumn("report_group", coalesce(trim(col("report_group")), lit("Unknown"))) \
    .withColumn("tender_type", coalesce(trim(col("tender_type")), lit("Unknown")))

print("✓ Data types converted and cleaned")
print()

# ============================================================================
# STEP 4: REMOVE DUPLICATES
# ============================================================================

print("STEP 4: Conservar líneas; no existe una clave única de línea validada.")
print("-" * 80)

before_dedup = df_typed.count()

df_dedup = df_typed
print("Filas conservadas; revisión de duplicados pendiente.")

after_dedup = df_dedup.count()
print("No se eliminan líneas iguales: pueden representar compras legítimas.")
print(f"  Remaining: {after_dedup:,} rows")
print()

# ============================================================================
# STEP 5: CREATE FLAGS AND DERIVED FIELDS
# ============================================================================

print("STEP 5: Creating flags and derived fields...")
print("-" * 80)

df_final = df_dedup \
    .withColumn("is_negative_total", when(col("total") < 0, 1).otherwise(0)) \
    .withColumn("is_negative_price", when(col("menu_price") < 0, 1).otherwise(0)) \
    .withColumn("is_zero_quantity", when(col("quantity") == 0, 1).otherwise(0)) \
    .withColumn("is_zero_price", when(col("menu_price") == 0, 1).otherwise(0)) \
    .withColumn("is_incomplete_store", when(col("store_number").isin(INCOMPLETE_STORES), 1).otherwise(0)) \
    .withColumn("net_total", lit(None).cast("double")) \
    .withColumn("net_item_price", col("quantity") * col("menu_price") - col("item_discount")) \
    .withColumn("price_per_unit",
        when(
            col("quantity") > 0,
            col("net_item_price") / col("quantity")
        ).otherwise(lit(None).cast("double"))
    ) \
    .withColumn("has_discount",
        when((col("total_discount") > 0) | (col("item_discount") > 0), 1).otherwise(0)
    ) \
    .withColumn("discount_pct",
        when(
            (col("quantity") > 0) & (col("menu_price") > 0),
            100 * col("item_discount") / (col("quantity") * col("menu_price"))
        ).otherwise(lit(None).cast("double"))
    ) \
    .withColumn("is_voided",
        when(
            (coalesce(col("order_voided"), lit(0)) != 0) |
            (coalesce(col("item_voided"), lit(0)) != 0) |
            (lower(trim(col("transaction_type"))) == "void"),
            1
        ).otherwise(0)
    ) \
    .withColumn("item_category",
        when(col("report_group").contains("Taco"), "Tacos")
        .when(col("report_group").contains("Burrito"), "Burritos")
        .when(col("report_group").contains("Beverage") | col("report_group").contains("Drink"), "Beverages")
        .when(col("report_group").contains("Side") | col("report_group").contains("Nacho") | col("report_group").contains("Chip"), "Sides")
        .when(col("report_group").contains("Quesadilla"), "Quesadillas")
        .when(col("report_group").contains("Condiment") | col("report_group").contains("Modifier"), "Modifiers")
        .when(col("report_group").contains("Paper"), "Packaging")
        .otherwise("Other")
    ) \
    .withColumn("is_online_order",
        when(lower(col("channel")).contains("online"), 1).otherwise(0)
    )

print("✓ Created all derived fields and flags")
print()

# ============================================================================
# STEP 6: SELECT FINAL COLUMNS
# ============================================================================

print("STEP 6: Selecting final columns...")
print("-" * 80)

final_columns = [
    # Identifiers
    "store_number", "check_number", "pos_code",
    # Date/Time
    "order_date", "year", "month", "day", "day_of_week", "day_name",
    "week_of_year", "is_weekend", "begin_time", "end_time", "begin_hour",
    "end_hour", "time_period",
    # Transaction
    "transaction_type", "channel", "tender_type",
    # Item
    "menu_item", "menu_item_ext_ref", "report_group", "item_category", "quantity",
    # Pricing
    "menu_price", "item_discount", "net_item_price", "price_per_unit",
    "total", "total_discount", "net_total", "tax",
    # Flags
    "has_discount", "discount_pct", "is_voided", "order_voided", "item_voided",
    "is_negative_total", "is_negative_price", "is_zero_quantity",
    "is_zero_price", "is_incomplete_store", "is_online_order",
    # Other
    "time_card", "source_file"
]

# Normalizar por ticket: venta reconstruida después del descuento, antes de impuestos.
# Dos convenciones observadas: Total = bruto; Total = bruto - descuento + impuesto.
# Solo publicar el neto de tickets que concilien; no asignar una regla por tienda.
from pyspark.sql import functions as F
keys = ["store_number", "order_date", "check_number"]
reconciliacion = (df_final.groupBy(*keys).agg(
    F.sum(F.col("quantity") * F.col("menu_price")).alias("gross_ticket"),
    F.max("total").alias("recorded_total"),
    F.max("total_discount").alias("ticket_discount"),
    F.max("tax").alias("ticket_tax"),
    F.countDistinct("total").alias("n_total"),
    F.countDistinct("total_discount").alias("n_discount"),
    F.countDistinct("tax").alias("n_tax"),
    F.max("is_voided").alias("any_void"),
    F.max(F.when(
        F.col("quantity").isNull() | F.col("menu_price").isNull() |
        F.col("total").isNull() | F.col("total_discount").isNull() |
        F.col("tax").isNull() | (F.col("quantity") < 0) |
        (F.col("item_discount") < 0) | (F.col("total_discount") < 0), 1
    ).otherwise(0)).alias("invalid_amount")
).withColumn("matches_gross",
    F.abs(F.col("recorded_total") - F.col("gross_ticket")) <= 0.02
).withColumn("matches_paid",
    F.abs(F.col("recorded_total") - (
        F.col("gross_ticket") - F.col("ticket_discount") + F.col("ticket_tax")
    )) <= 0.02
).withColumn("net_total_status",
    F.when(F.col("any_void") != 0, "review_void")
    .when((F.col("invalid_amount") != 0) | (F.col("n_total") != 1) |
          (F.col("n_discount") != 1) | (F.col("n_tax") != 1), "review_amounts")
    .when(F.col("gross_ticket") - F.col("ticket_discount") < -0.02, "review_negative_net")
    .when(F.col("matches_gross") | F.col("matches_paid"), "reconciled")
    .otherwise("review_unreconciled")
).withColumn("net_total",
    F.when(F.col("net_total_status") == "reconciled",
        F.round(F.col("gross_ticket") - F.col("ticket_discount"), 2))
))
df_final = df_final.drop("net_total").join(
    reconciliacion.select(*keys, "net_total", "net_total_status"), keys, "left"
)
final_columns.append("net_total_status")
df_output = df_final.select(final_columns)


print(f"✓ Selected {len(final_columns)} columns")
print()

# Validaciones antes de publicar: no modifican la tabla existente.
invalid_keys = df_output.filter(
    col("store_number").isNull() | col("check_number").isNull() |
    col("order_date").isNull()
).limit(1).count()
assert invalid_keys == 0, "Hay claves de ticket faltantes; no se publica."

multiple_sources = (df_output
    .groupBy("store_number", "order_date", "check_number")
    .agg(countDistinct("source_file").alias("n_archivos"))
    .filter(col("n_archivos") > 1).limit(1).count())
assert multiple_sources == 0, "Hay tickets en varios archivos; revisar antes de publicar."

# Eliminar fechas inválidas se registra explícitamente en el informe.
invalid_dates = df_clean.filter(to_date(col("date"), "M/d/yyyy").isNull()).count()
print(f"Filas excluidas por fecha inválida: {invalid_dates}")

# ============================================================================
# STEP 7: DROP EXISTING TABLE (if any)
# ============================================================================

print("STEP 7: Preparing to create table...")
print("-" * 80)

assert DELTA_TABLE_NAME == "workspace.default.taco_bueno_clean"
print("Se actualizará la tabla definitiva sin DROP TABLE.")
print()

# ============================================================================
# STEP 8: WRITE AS MANAGED DELTA TABLE
# ============================================================================

print("STEP 8: Writing Delta table (managed by Databricks)...")
print("-" * 80)

# Write as a managed table (Databricks handles storage)
df_output.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .option("overwriteSchema", "true") \
    .saveAsTable(DELTA_TABLE_NAME)

print(f"✓ Table created: {DELTA_TABLE_NAME}")
print()

# ============================================================================
# STEP 9: ADD TABLE DESCRIPTION
# ============================================================================

print("STEP 9: Adding table description...")
print("-" * 80)

spark.sql(f"""
    COMMENT ON TABLE {DELTA_TABLE_NAME} IS 
    'Taco Bueno: base EDA. net_total reconstruido sin impuestos, solo net_total_status=reconciled. Cobertura según STEP 11. 
    Partitioned by year/month. 
    Use is_incomplete_store=0 to exclude stores with partial data (#5263, #5898).
    Created: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
""")

print("✓ Table description added")
print()

# ============================================================================
# STEP 10: OPTIMIZE TABLE
# ============================================================================

print("STEP 10: Optimizing table...")
print("-" * 80)

spark.sql(f"OPTIMIZE {DELTA_TABLE_NAME}")
spark.sql(f"ANALYZE TABLE {DELTA_TABLE_NAME} COMPUTE STATISTICS")

print("✓ Table optimized and statistics computed")
print()

# ============================================================================
# STEP 11: SUMMARY STATISTICS
# ============================================================================

print("STEP 11: Summary statistics...")
print("-" * 80)

# Una fila por ticket para no sumar sus importes por cada producto.
claves_ticket = ["store_number", "order_date", "check_number"]

tickets_revision = (
    spark.table(DELTA_TABLE_NAME)
    .filter(col("is_incomplete_store") == 0)
    .groupBy(*claves_ticket)
    .agg(
        countDistinct("total").alias("versiones_total"),
        countDistinct("total_discount").alias("versiones_descuento"),
        countDistinct("tax").alias("versiones_impuesto"),
        max("total").alias("total_registrado"),
        max("total_discount").alias("descuento_ticket"),
        max("tax").alias("impuesto"),
        max("has_discount").alias("con_descuento")
    )
)

# Revisar tickets cuyos importes no son constantes entre sus filas.
inconsistentes = tickets_revision.filter(
    (col("versiones_total") != 1) |
    (col("versiones_descuento") != 1) |
    (col("versiones_impuesto") != 1)
)

display(inconsistentes)

# Resumen provisional: solo tickets con importes consistentes.
tickets_consistentes = tickets_revision.filter(
    (col("versiones_total") == 1) &
    (col("versiones_descuento") == 1) &
    (col("versiones_impuesto") == 1)
)

display(tickets_consistentes.agg(
    count("*").alias("tickets_consistentes"),
    round(sum("total_registrado"), 2).alias("total_registrado"),
    round(sum("descuento_ticket"), 2).alias("descuento_de_ticket"),
    sum("con_descuento").alias("tickets_con_descuento")
))

print()

# ============================================================================
# STEP 12: SHOW TABLE LOCATION
# ============================================================================

print("STEP 12: Table information...")
print("-" * 80)

table_info = spark.sql(f"DESCRIBE EXTENDED {DELTA_TABLE_NAME}")
location = table_info.filter(col("col_name") == "Location").select("data_type").first()

if location:
    print(f"📁 Table location: {location[0]}")
else:
    print("📁 Table location: Managed by Databricks")

print()

# ============================================================================
# COMPLETION
# ============================================================================

print("=" * 80)
print("✅ DELTA TABLE CREATION COMPLETE!")
print("=" * 80)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()
print(f"📊 Table name: {DELTA_TABLE_NAME}")
print()
print("🎯 Your team can now use:")
print(f"   SQL: SELECT * FROM {DELTA_TABLE_NAME} WHERE month = 1")
print(f"   Python: df = spark.table('{DELTA_TABLE_NAME}')")
print(f"   Pandas: df = spark.table('{DELTA_TABLE_NAME}').filter('month = 1').toPandas()")
print()
print("📋 Next steps:")
print("   1. Base EDA publicada. Usar net_total solo con net_total_status=reconciled y una vez por ticket.")
print("   2. Share table name with your team")
print("   3. Create analysis notebooks")
print("   4. Export to CSV if needed for R")
print("=" * 80)
# INFORME FINAL DE CALIDAD (guardar junto al notebook)
base_publicada = spark.table(DELTA_TABLE_NAME)
display(base_publicada.agg(
    count("*").alias("filas"),
    min("order_date").alias("fecha_inicial"),
    max("order_date").alias("fecha_final"),
    sum(when(col("is_voided") == 1, 1).otherwise(0)).alias("lineas_anuladas"),
    sum(when(col("quantity") <= 0, 1).otherwise(0)).alias("cantidad_no_positiva"),
    sum(when(col("quantity").isNull(), 1).otherwise(0)).alias("cantidad_faltante"),
    sum(when(col("discount_pct") < 0, 1).otherwise(0)).alias("descuento_negativo"),
    sum(when(col("discount_pct") > 100.01, 1).otherwise(0)).alias("descuento_mayor_100"),
    sum(when(col("net_total").isNotNull(), 1).otherwise(0)).alias("lineas_con_neto_conciliado")
))
print("PART1 ejecutado. Revisar informe; los totales del STEP 11 son descriptivos, no ingresos netos.")

# Cobertura de conciliación: tickets, NO líneas.
display(reconciliacion.groupBy("net_total_status").count())
# Vista de consulta a nivel ticket; evita sumar el mismo neto por producto.
# El filtro de tiendas incompletas se aplica en cada consulta según el README.

TACO BUENO - CREATE CLEAN DELTA TABLE
Start time: 2026-10-08 00:23:51

📁 Input path: /Volumes/workspace/default/taco_bueno_raw/TB_Daily_TLD_*-2026-m.csv
📊 Table name: workspace.default.taco_bueno_clean

STEP 1: Reading raw CSV files...
--------------------------------------------------------------------------------
✓ Loaded 42,253,092 rows

STEP 2: Removing header rows...
--------------------------------------------------------------------------------
✓ Removed 0 header rows
  Remaining: 42,253,092 rows

STEP 3: Converting data types and cleaning...
--------------------------------------------------------------------------------
✓ Data types converted and cleaned

STEP 4: Conservar líneas; no existe una clave única de línea validada.
--------------------------------------------------------------------------------
Filas conservadas; revisión de duplicados pendiente.
No se eliminan líneas iguales: pueden representar compras legítimas.
  Remaining: 42,253,092 rows

STEP 5: Creating flags 

store_number,order_date,check_number,versiones_total,versiones_descuento,versiones_impuesto,total_registrado,descuento_ticket,impuesto,con_descuento


tickets_consistentes,total_registrado,descuento_de_ticket,tickets_con_descuento
6012195,8.743520905E7,1574701.87,432954



STEP 12: Table information...
--------------------------------------------------------------------------------
📁 Table location: 

✅ DELTA TABLE CREATION COMPLETE!
End time: 2026-10-08 00:28:29

📊 Table name: workspace.default.taco_bueno_clean

🎯 Your team can now use:
   SQL: SELECT * FROM workspace.default.taco_bueno_clean WHERE month = 1
   Python: df = spark.table('workspace.default.taco_bueno_clean')
   Pandas: df = spark.table('workspace.default.taco_bueno_clean').filter('month = 1').toPandas()

📋 Next steps:
   1. Base EDA publicada. Usar net_total solo con net_total_status=reconciled y una vez por ticket.
   2. Share table name with your team
   3. Create analysis notebooks
   4. Export to CSV if needed for R


filas,fecha_inicial,fecha_final,lineas_anuladas,cantidad_no_positiva,cantidad_faltante,descuento_negativo,descuento_mayor_100,lineas_con_neto_conciliado
42253092,2025-12-31,2026-09-27,157162,74737,0,3,0,41716603


PART1 ejecutado. Revisar informe; los totales del STEP 11 son descriptivos, no ingresos netos.


net_total_status,count
review_void,61812
review_negative_net,2
reconciled,6002815
review_unreconciled,2148
review_amounts,1
